# niftyhedge — full Colab run (from scratch)

**Before this notebook**

1. On your laptop, Gate 1a must be done (`nifty_options_raw.parquet` ~1.1 GB).
2. Upload to Drive **exactly**:
```text
My Drive/Vol-Surfaces/data/raw/nifty_options_raw.parquet
My Drive/Vol-Surfaces/data/raw/download_log.csv
```
3. Runtime → Change runtime type → **GPU (T4)**.

Stop at every gate. Do not skip ahead if a check fails.

## Cell 1 — GPU check

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU"

## Cell 2 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

!ls -lh "/content/drive/MyDrive/Vol-Surfaces/data/raw/"

## Cell 3 — Clone repo + link raw data

Must show `nifty_options_raw.parquet` ~1.1G. If missing, stop and finish the Drive upload.

In [ ]:
import os
from datetime import datetime

REPO = "/content/Vol-Surfaces"
DRIVE_RAW = "/content/drive/MyDrive/Vol-Surfaces/data/raw"
RUN = f"/content/drive/MyDrive/Vol-Surfaces/runs/{datetime.now().strftime('%Y%m%d')}"
os.makedirs(RUN, exist_ok=True)
os.environ["NH_RUN"] = RUN

# fresh clone each session (safe)
!rm -rf {REPO}
!git clone https://github.com/Adya6714/ddpm-vol-hedging.git {REPO}
%cd {REPO}

!rm -rf data/raw
os.makedirs("data", exist_ok=True)
!ln -s "{DRIVE_RAW}" data/raw

!ls -lh data/raw/nifty_options_raw.parquet data/raw/download_log.csv
assert os.path.exists("data/raw/nifty_options_raw.parquet"), \
    "Upload nifty_options_raw.parquet to MyDrive/Vol-Surfaces/data/raw/ first"
print("RUN dir:", RUN)

## Cell 4 — Install package

In [ ]:
!pip install -U pip -q
!pip install -e ".[dev]" -q
!python -c "import niftyhedge, torch; print('niftyhedge', niftyhedge.__version__, 'cuda', torch.cuda.is_available())"

## Cell 5 — Gate 0 (tests + build check)

Expect 26 passed. Legacy audit needs `models/rl/` from the repo (already in the clone).

In [ ]:
!pytest -q
!python scripts/check_build.py
!python legacy/audit_env.py . 50 | tee "$NH_RUN/legacy_audit.txt"

## Cell 6 — Stage 1: build SVI surfaces

This can take **30–90+ minutes**. Do not close the tab. Progress prints every 100 days.

In [ ]:
!python scripts/01_build_surfaces.py --config configs/data.yaml

import pandas as pd
card = pd.read_csv("tables/dataset_card.csv")
print(card.to_string(index=False))
assert card.loc[0, "nan_frac"] == 0.0, "NaNs in surfaces"
assert str(card.loc[0, "last"]) >= "2026-03-01", "last surface date too early"
print("Gate 1b (surfaces): PASS")

## Cell 7 — Stage 1b: walk-forward folds

In [ ]:
!python scripts/02_make_folds.py --config configs/folds.yaml
!ls data/folds
!cat data/folds/fold_manifest.json

## Cell 8 — Save Stage 1 artefacts to Drive

Surfaces/folds live on the ephemeral Colab disk unless you copy them.

In [ ]:
import shutil
from pathlib import Path

dest = Path("/content/drive/MyDrive/Vol-Surfaces")
for name in ["data/surfaces", "data/folds", "tables"]:
    src = Path(name)
    out = dest / name
    if out.exists():
        shutil.rmtree(out)
    shutil.copytree(src, out)
    print("copied", src, "->", out)

!ls -lh /content/drive/MyDrive/Vol-Surfaces/data/surfaces/
!ls /content/drive/MyDrive/Vol-Surfaces/data/folds/

## Cell 9 — Stage 2: fit generators (GPU)

MVP: all worlds on all folds; DDPM gets more seeds on the main fold.
If a Colab session dies, re-run from Cell 2–4, restore surfaces/folds from Drive, then continue.

In [ ]:
# If reconnecting mid-run, uncomment to restore Stage 1 from Drive:
# !rm -rf data/surfaces data/folds tables
# !cp -a /content/drive/MyDrive/Vol-Surfaces/data/surfaces data/
# !cp -a /content/drive/MyDrive/Vol-Surfaces/data/folds data/
# !cp -a /content/drive/MyDrive/Vol-Surfaces/tables .

!python scripts/10_fit_generators.py --config configs/gen/gbm.yaml --folds all --seeds 0
!python scripts/10_fit_generators.py --config configs/gen/heston.yaml --folds all --seeds 0
!python scripts/10_fit_generators.py --config configs/gen/pcavar.yaml --folds all --seeds 0
!python scripts/10_fit_generators.py --config configs/gen/bootstrap.yaml --folds all --seeds 0
!python scripts/10_fit_generators.py --config configs/gen/ddpm.yaml --folds all --seeds 0,1,2
!python scripts/10_fit_generators.py --config configs/gen/ddpm.yaml --folds 5 --seeds 3,4

## Cell 10 — Generate paths + generator quality

In [ ]:
!python scripts/11_generate_paths.py --worlds gbm,heston,pcavar,bootstrap,ddpm --folds all \
    --n-train 200000 --n-test 100000

!python scripts/12_eval_generators.py --config configs/experiments/e5_gen_quality.yaml

import pandas as pd
print(pd.read_csv("tables/world_stats.csv").to_string(index=False))
print(pd.read_csv("tables/gen_quality.csv").head(20).to_string(index=False))

### Gate 2 checklist (read the tables above)
- [ ] DDPM mean ATM near real for same starts
- [ ] No NaN / non-positive vols
- [ ] Quality metrics written

Copy models + paths to Drive before Stage 3 if the session might die.

In [ ]:
import shutil
from pathlib import Path
dest = Path("/content/drive/MyDrive/Vol-Surfaces")
for name in ["models/gen", "data/paths", "tables"]:
    src = Path(name)
    if not src.exists():
        print("skip missing", src)
        continue
    out = dest / name
    if out.exists():
        shutil.rmtree(out)
    shutil.copytree(src, out)
    print("copied", src, "->", out)

## Cell 11 — Gate 3: W0 deep-hedge sanity (must pass)

In [ ]:
!python scripts/20_train_hedgers.py --config configs/hedge/deep_hedge.yaml \
    --world gbm --fold 5 --instruments I1 --contract configs/contracts/doc_090.yaml \
    --cost-bps 0 --seeds 0

!python scripts/check_build.py --only A14

## Cell 12 — Stage 3: train deep hedgers (long)

MVP shortcut below uses worlds `gbm,heston,pcavar,ddpm` and instruments `I1,I2` only.
Full paper: add `bootstrap` and `I3`.

In [ ]:
WORLDS = "gbm heston pcavar ddpm"   # add: bootstrap
INSTRS = "I1 I2"                   # add: I3

for W in WORLDS.split():
    for I in INSTRS.split():
        print("====", W, I, "fold 5 ====")
        !python scripts/20_train_hedgers.py --config configs/hedge/deep_hedge.yaml \
            --world {W} --fold 5 --instruments {I} --contract configs/contracts/doc_090.yaml \
            --seeds 0,1,2,3,4
        print("====", W, I, "folds 0-4 ====")
        !python scripts/20_train_hedgers.py --config configs/hedge/deep_hedge.yaml \
            --world {W} --folds 0,1,2,3,4 --instruments {I} --contract configs/contracts/doc_090.yaml \
            --seeds 0,1,2

# PPO ablation
!python scripts/20_train_hedgers.py --config configs/hedge/ppo.yaml \
    --world ddpm --fold 5 --instruments I1 --contract configs/contracts/doc_090.yaml \
    --seeds 0,1,2,3,4

In [ ]:
import shutil
from pathlib import Path
dest = Path("/content/drive/MyDrive/Vol-Surfaces/models")
dest.mkdir(parents=True, exist_ok=True)
for name in ["models/hedge", "models/hedge_ppo"]:
    src = Path(name)
    if not src.exists():
        continue
    out = dest / src.name
    if out.exists():
        shutil.rmtree(out)
    shutil.copytree(src, out)
    print("copied", src, "->", out)

## Cell 13 — Stage 4: cross-world + real backtest + tables

In [ ]:
!python scripts/21_cross_world_eval.py --config configs/experiments/e7_cross_world.yaml
!python scripts/22_real_backtest.py --config configs/experiments/e8_real_backtest.yaml
!python scripts/30_sensitivity.py --config configs/experiments/e10_sensitivity.yaml
!python scripts/40_make_tables_figures.py --run "$NH_RUN"

!ls -la tables/ | head -40
!cp -a tables figures "$NH_RUN/" 2>/dev/null || true
!cp -a tables /content/drive/MyDrive/Vol-Surfaces/
!cp -a figures /content/drive/MyDrive/Vol-Surfaces/ 2>/dev/null || true
print("done — tables on Drive and in", __import__("os").environ.get("NH_RUN"))

## Done / reconnect cheat sheet

After a disconnect: remount Drive → clone → install → symlink `data/raw` → copy `data/surfaces`, `data/folds`, `models/`, `data/paths` back from Drive → resume the cell you were on.

Scripts skip finished seeds when `manifest.json` already matches.